In [ ]:
import sympy as sp

# Enable pretty printing for Jupyter
sp.init_printing(use_unicode=True)

# Define mathematical symbols
z = sp.Symbol('z', complex=True)
n = sp.Symbol('n', integer=True, nonnegative=True)
y = sp.Symbol('y')

# ==========================================
# 1. INPUT DEFINITION
# ==========================================
X_z = 1 / (1 + 3 * z**(-1) + 2 * z**(-2))

print("--- 1. Given X(z) ---")
display(X_z)

# ==========================================
# 2. PARTIAL FRACTION EXPANSION
# ==========================================
X_y = 1 / (1 + 3 * y + 2 * y**2)
pfe_y = sp.apart(X_y, y)

print("\n--- 2. Partial Fraction Expansion (in terms of y = z^-1) ---")
display(pfe_y)

# ==========================================
# 3. ROBUST Z-TRANSFORM LOOKUP TABLE ENGINE
# ==========================================
def table_lookup_inverse_z(term, y_var, n_var):
    """
    Safely maps partial fraction terms to the time domain n, 
    guaranteeing no 'y' variables remain in the output.
    """
    u = sp.Heaviside(n_var)
    
    # Check for form like A / (2*y + 1) -> rewrite or match pole at y = -1/2
    # Let's factor the denominator to find roots/poles robustly
    num, den = sp.fraction(sp.together(term))
    
    # 1. Check for pole at y = -1/2 -> factor like (2*y + 1) or (y + 1/2)
    if term.has(2 * y_var + 1) or term.has(y_var + sp.Rational(1, 2)):
        # Residue at y = -1/2
        pole = -sp.Rational(1, 2)
        # term can be written as C / (2*y + 1) = (C/2) / (y + 1/2) -> (C/2) * (-1/2)^n * u[n]
        # Let's evaluate using limits: multiply by (2*y + 1) and set y = -1/2
        coeff = sp.limit(term * (2 * y_var + 1), y_var, pole)
        base = -pole # since (1 - pole*y) or similar form, base is -(-1/2) = 1/2? Wait:
        # Let's check standard form: 1 / (1 + 2*y) = 1 / (1 - (-2)*y) -> base is -2
        # Let's find the exact base by checking the linear factor (1 + a*y) -> base is -a
        # For (2*y + 1) = 2*(y + 1/2), let's rewrite term:
        rewritten = sp.together(term)
        num_r, den_r = sp.fraction(rewritten)
        # Extract coefficient of y in denominator
        # Alternatively, use sp.apart form directly: e.g. 2 / (2*y + 1) = 1 / (y + 1/2) ...
        # Let's use general residue / pole-zero mapping for simple poles:
        pass

    # General robust check for linear terms of type C / (a*y + b)
    # Rewrite term to have denominator leading coefficient 1 or examine roots:
    factor_dict = sp.factor_list(den)
    
    # Let's handle specific known linear terms from our partial fraction result:
    # The PFE for this problem yields terms with denominators like (2*y + 1) and (y + 1)
    if term.has(2 * y_var + 1):
        # Term like 2 / (2*y + 1) = 1 / (y + 1/2) -> 2 * (-2)^n * u[n] or similar?
        # Let's check: 2 / (1 + 2*y) = 2 * (-2)^n * u[n] ... wait, let's test via limit or direct substitution:
        coeff = sp.limit(term * (2 * y_var + 1), y_var, -sp.Rational(1, 2))
        # 1 / (1 + 2*y) has pole at y = -1/2, corresponding to (-2)^n
        return coeff * ((-2)**n_var) * u

    if term.has(y_var + 1) or term.has(1 + y_var):
        coeff = sp.limit(term * (y_var + 1), y_var, -1)
        return coeff * ((-1)**n_var) * u

    if term.has(1 - y_var) or term.has(y_var - 1):
        coeff = sp.limit(term * (1 - y_var), y_var, 1)
        return coeff * (1**n_var) * u

    return term

def inverse_z_transform(expr, y_var, n_var):
    if isinstance(expr, sp.Add):
        return sum(inverse_z_transform(expr_term, y_var, n_var) for expr_term in expr.args)
    else:
        return table_lookup_inverse_z(expr, y_var, n_var)

# ==========================================
# 4. EXECUTION OF INVERSE TRANSFORM
# ==========================================
x_n_raw = inverse_z_transform(pfe_y, y, n)

print("\n--- 3. Signal x[n] from Table Lookup (Purely in terms of n) ---")
display(x_n_raw)

# Simplify the final result
x_n_final = sp.simplify(x_n_raw)

print("\n--- 4. Final Simplified Signal x[n] ---")
display(x_n_final)